<a href="https://colab.research.google.com/github/NeoNest/Soy-Bean-Analytic-for-food-security/blob/main/ConvNext.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1. Setup

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import pandas as pd
import os
import numpy as np
from torchvision.ops import StochasticDepth
from typing import List
from PIL import Image
from glob import glob
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

##2. Loading Data

In [4]:
data_dir = '/content/drive/My Drive/lomba/An India soyabean leaf dataset/Soyabean leaf'

In [5]:
import os

# Memeriksa isi dari folder utama di Drive untuk memastikan path-nya tepat
base_path = '/content/drive/My Drive/lomba'
try:
    print("Isi folder 'lomba':", os.listdir(base_path))
    sub_path = os.path.join(base_path, 'An India soyabean leaf dataset')
    if os.path.exists(sub_path):
        print("Isi folder 'An India soyabean leaf dataset':", os.listdir(sub_path))
    else:
        print("Folder 'An India soyabean leaf dataset' tidak ditemukan di path tersebut.")
except Exception as e:
    print("Error saat membaca direktori:", e)

Isi folder 'lomba': ['An India soyabean leaf dataset']
Isi folder 'An India soyabean leaf dataset': ['Soyabean leaf']


In [6]:
# Split dataset into training, validation, and test sets using updated data_dir
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

# Confirm directories existence and print contents
for name, path in [('Train', train_dir), ('Val', val_dir), ('Test', test_dir)]:
    if os.path.exists(path):
        print(f"{name} directory exists. Subfolders/Classes:", os.listdir(path))
    else:
        print(f"{name} directory DOES NOT exist at {path}")

Train directory DOES NOT exist at /content/drive/My Drive/lomba/An India soyabean leaf dataset/Soyabean leaf/train
Val directory DOES NOT exist at /content/drive/My Drive/lomba/An India soyabean leaf dataset/Soyabean leaf/val
Test directory DOES NOT exist at /content/drive/My Drive/lomba/An India soyabean leaf dataset/Soyabean leaf/test


In [7]:
import os

# Memetakan struktur direktori di dalam 'Soyabean leaf'
for root, dirs, files in os.walk('/content/drive/My Drive/lomba'):
    level = root.replace('/content/drive/My Drive/lomba', '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subdirs = dirs[:5]  # Batasi tampilan agar tidak terlalu panjang
    for d in subdirs:
        print(f'{indent}    {d}/')

lomba/
    An India soyabean leaf dataset/
    An India soyabean leaf dataset/
        Soyabean leaf/
        Soyabean leaf/
            5.root_images/
            1.Healthy/
            2.Vein Necrosis/
            4.Septoria_Brown_Spot/
            3.Dry_leaf/
            5.root_images/
            1.Healthy/
                Multi_Healthy_leaf/
                Single_Healthy_leaf/
                Multi_Healthy_leaf/
                Single_Healthy_leaf/
            2.Vein Necrosis/
                Multi_Vein Necrosis/
                Single_Vein Necrosis Virus/
                Multi_Vein Necrosis/
                Single_Vein Necrosis Virus/
            4.Septoria_Brown_Spot/
                Multi_Septoria_Brown_spot/
                Single_Septoria_Brown_spot/
                Multi_Septoria_Brown_spot/
                Single_Septoria_Brown_spot/
            3.Dry_leaf/
                Single_Dry_leaf/
                Multi_Dry_leaf/
                Single_Dry_leaf/
                Mul

In [8]:
import os

# Mengabaikan file non-folder, folder sistem, dan folder raw data 'Leaf Images All (Raw)'
classes = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
    and not d.startswith('.')
    and d != 'Leaf Images All (Raw)'
])
print("Detected classes:", classes)

Detected classes: ['1.Healthy', '2.Vein Necrosis', '3.Dry_leaf', '4.Septoria_Brown_Spot', '5.root_images', '6.Bacterial leaf Blight']


In [9]:
# Menampilkan jumlah file untuk kelas pertama
first_class_dir = os.path.join(data_dir, classes[0])
first_class_files = []
for root, dirs, files in os.walk(first_class_dir):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            first_class_files.append(f)

print(f'No. of images for {classes[0]}:', len(first_class_files))
print(first_class_files[:5])

No. of images for 1.Healthy: 288
['IMG_20230917_142039_1.jpg', 'IMG_20230917_142040.jpg', 'IMG_20230917_142041_1.jpg', 'IMG_20230917_142041.jpg', 'IMG_20230917_142047_1.jpg']


In [10]:
# Menampilkan jumlah file untuk kelas kedua jika ada
if len(classes) > 1:
    second_class_dir = os.path.join(data_dir, classes[1])
    second_class_files = []
    for root, dirs, files in os.walk(second_class_dir):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                second_class_files.append(f)
    print(f'No. of images for {classes[1]}:', len(second_class_files))
    print(second_class_files[:5])
else:
    print("Hanya terdeteksi 1 kelas.")

No. of images for 2.Vein Necrosis: 138
['IMG_20230917_142135.jpg', 'IMG_20230917_142136.jpg', 'IMG_20230917_142136_1.jpg', 'IMG_20230917_142136_2.jpg', 'IMG_20230917_142137.jpg']


In [14]:
# Menghitung bobot kelas secara dinamis dari subset training
from collections import Counter
import torch

# Pastikan train_dataset sudah didefinisikan di sel sebelumnya
try:
    # Jika train_dataset dibungkus oleh DatasetWithTransform
    if hasattr(train_dataset, 'subset'):
        train_indices = train_dataset.subset.indices
    else:
        train_indices = train_dataset.indices

    train_targets = [full_dataset.targets[idx] for idx in train_indices]
    target_counts = Counter(train_targets)

    num_samples = [target_counts[i] for i in range(len(classes))]
    total_train = sum(num_samples)

    # Menghitung bobot kelas (terbalik secara proporsional dengan frekuensi kelas)
    weights = [total_train / count if count > 0 else 1.0 for count in num_samples]
    class_weights = torch.tensor(weights, dtype=torch.float).to(device)
    print("Class weights:", class_weights)
except NameError as e:
    print("Error: Pastikan Anda menjalankan sel pembuatan dataset (Section 3) terlebih dahulu sebelum menjalankan sel ini!\n", e)

Error: Pastikan Anda menjalankan sel pembuatan dataset (Section 3) terlebih dahulu sebelum menjalankan sel ini!
 name 'train_dataset' is not defined


##3. Preparing Dataset and DataLoader

In [15]:
# Define transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=10, translate=(0.05,0.05)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

val_and_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

In [16]:
from torch.utils.data import random_split

# Membuat subclass kustom dari ImageFolder agar hanya membaca kelas yang kita inginkan
class CustomImageFolder(ImageFolder):
    def find_classes(self, directory):
        # Hanya menggunakan kelas yang terdaftar di variabel 'classes' yang sudah kita saring
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx

# Memuat seluruh dataset dari data_dir utama menggunakan kelas kustom kita
full_dataset = CustomImageFolder(data_dir)

# Menentukan ukuran pembagian (80% Train, 10% Val, 10% Test)
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

# Split dataset secara acak menggunakan generator dengan seed agar konsisten
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)

# Menerapkan transformasi secara spesifik pada masing-masing subset dataset
class DatasetWithTransform(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset = DatasetWithTransform(train_dataset, transform=train_transform)
val_dataset = DatasetWithTransform(val_dataset, transform=val_and_test_transform)
test_dataset = DatasetWithTransform(test_dataset, transform=val_and_test_transform)

# Membuat DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Dataset berhasil dibagi! Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Dataset berhasil dibagi! Train: 940, Val: 117, Test: 119


##4. Defining and Choosing Model

In [17]:
class LayerScaler(nn.Module):
    def __init__(self, init_value: float, dimensions: int):
        super().__init__()
        self.gamma = nn.Parameter(init_value * torch.ones((dimensions)),
                                    requires_grad=True)

    def forward(self, x):
        return self.gamma[None,...,None,None] * x

class ConvNormAct(nn.Sequential):
    # A little util layer composed by (conv) -> (norm) -> (act) layers.
    def __init__(
        self,
        in_features: int,
        out_features: int,
        kernel_size: int,
        norm = nn.BatchNorm2d,
        act = nn.ReLU,
        **kwargs
    ):
        super().__init__(
            nn.Conv2d(
                in_features,
                out_features,
                kernel_size=kernel_size,
                padding=kernel_size // 2,
                **kwargs
            ),
            norm(out_features),
            act(),
        )

class BottleNeckBlock(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        expansion: int = 4,
        drop_p: float = .0,
        layer_scaler_init_value: float = 1e-6,
    ):
        super().__init__()
        expanded_features = out_features * expansion
        self.block = nn.Sequential(
            # narrow -> wide (with depth-wise and bigger kernel)
            nn.Conv2d(
                in_features, in_features, kernel_size=7, padding=3, bias=False, groups=in_features
            ),
            # GroupNorm with num_groups=1 is the same as LayerNorm but works for 2D data
            nn.GroupNorm(num_groups=1, num_channels=in_features),
            # wide -> wide
            nn.Conv2d(in_features, expanded_features, kernel_size=1),
            nn.GELU(),
            # wide -> narrow
            nn.Conv2d(expanded_features, out_features, kernel_size=1),
        )
        self.layer_scaler = LayerScaler(layer_scaler_init_value, out_features)
        self.drop_path = StochasticDepth(drop_p, mode="batch")


    def forward(self, x: Tensor) -> Tensor:
        res = x
        x = self.block(x)
        x = self.layer_scaler(x)
        x = self.drop_path(x)
        x += res
        return x

class ConvNexStage(nn.Sequential):
    def __init__(
        self, in_features: int, out_features: int, depth: int, **kwargs
    ):
        super().__init__(
            # add the downsampler
            nn.Sequential(
                nn.GroupNorm(num_groups=1, num_channels=in_features),
                nn.Conv2d(in_features, out_features, kernel_size=2, stride=2)
            ),
            *[
                BottleNeckBlock(out_features, out_features, **kwargs)
                for _ in range(depth)
            ],
        )

class ConvNextStem(nn.Sequential):
    def __init__(self, in_features: int, out_features: int):
        super().__init__(
            nn.Conv2d(in_features, out_features, kernel_size=4, stride=4),
            nn.BatchNorm2d(out_features)
        )

class ConvNextEncoder(nn.Module):
    def __init__(
        self,
        in_channels: int,
        stem_features: int,
        depths: List[int],
        widths: List[int],
        drop_p: float = .0,
    ):
        super().__init__()
        self.stem = ConvNextStem(in_channels, stem_features)

        in_out_widths = list(zip(widths, widths[1:]))
        # create drop paths probabilities (one for each stage)
        drop_probs = [x.item() for x in torch.linspace(0, drop_p, sum(depths))]

        self.stages = nn.ModuleList(
            [
                ConvNexStage(stem_features, widths[0], depths[0], drop_p=drop_probs[0]),
                *[
                    ConvNexStage(in_features, out_features, depth, drop_p=drop_p)
                    for (in_features, out_features), depth, drop_p in zip(
                        in_out_widths, depths[1:], drop_probs[1:]
                    )
                ],
            ]
        )


    def forward(self, x):
        x = self.stem(x)
        for stage in self.stages:
            x = stage(x)
        return x

class ClassificationHead(nn.Sequential):
    def __init__(self, num_channels: int, num_classes: int = 2):
        super().__init__(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(1),
            nn.LayerNorm(num_channels),
            nn.Linear(num_channels, num_classes)
        )

class ConvNext(nn.Sequential):
    def __init__(self,
                 in_channels: int,
                 stem_features: int,
                 depths: List[int],
                 widths: List[int],
                 drop_p: float = .0,
                 num_classes: int = 2):
        super().__init__()
        self.encoder = ConvNextEncoder(in_channels, stem_features, depths, widths, drop_p)
        self.head = ClassificationHead(widths[-1], num_classes)


In [18]:
# Initializing model with the correct number of classes dynamically
num_classes = len(classes)
model = ConvNext(in_channels=3, stem_features=64, depths=[3,4,6,4], widths=[256, 512, 1024, 2048], num_classes=num_classes)

In [19]:
if torch.cuda.is_available():
    model.cuda()

##5. Testing Model

In [20]:
x = torch.rand(1, 3, 224, 224).to(device)
model(x).shape

torch.Size([1, 6])

##6. Defining Main Training

In [21]:
# Hyper-parameters
num_epochs = 30
learning_rate = 0.001
patience = 10

In [25]:
# Loss and optimizer
# Check if class_weights is defined, otherwise fall back to None
loss_weight = class_weights if 'class_weights' in globals() else None
if loss_weight is None:
    print("Warning: class_weights is not defined. Training without class weighting.")
else:
    print("Training using calculated class_weights.")

criterion = nn.CrossEntropyLoss(weight=loss_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

curr_lr = learning_rate
total_step = len(train_loader)

# Early stopping parameters
early_stopping_counter = 0
best_loss = float('inf')

# For updating learning rate
def update_lr(optimizer, lr):
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr


for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 100 == 0:
            print ("Epoch [{}/{}], Step [{}/{}] Loss: {:.4f}"
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))

    # Decay learning rate
    if (epoch+1) % 20 == 0:
        curr_lr /= 3
        update_lr(optimizer, curr_lr)

    # Calculate validation loss
    with torch.no_grad():
        val_loss = 0
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels)
        val_loss /= len(val_loader)

        # Save the model if the validation loss is the best observed yet
        if val_loss < best_loss:
            print(f'Saving model with validation loss of {val_loss:.4f}...')
            torch.save(model.state_dict(), 'best_model.pth')
            best_loss = val_loss

    # Early stopping if overfitting
    if early_stopping_counter >= patience:
        print(f'Validation loss has not improved for {patience} epochs. Early stopping...')
        break
    elif val_loss < best_loss:
        best_loss = val_loss
        early_stopping_counter = 0

Saving model with validation loss of 1.2742...
Saving model with validation loss of 1.0574...
Saving model with validation loss of 0.8960...
Saving model with validation loss of 0.7167...
Saving model with validation loss of 0.5711...
Saving model with validation loss of 0.5528...
Saving model with validation loss of 0.5341...
Saving model with validation loss of 0.5261...
Saving model with validation loss of 0.4955...
Saving model with validation loss of 0.4905...
Saving model with validation loss of 0.4654...
Saving model with validation loss of 0.4360...
Saving model with validation loss of 0.4161...
Saving model with validation loss of 0.3813...
Saving model with validation loss of 0.3334...
Saving model with validation loss of 0.3316...


##7. Testing

In [26]:
# Load the saved model checkpoint
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint)

model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Model accuracy on test images: {} %'.format(100 * correct / total))

Model accuracy on test images: 89.91596638655462 %
